# Link Prediction -- Setup (Experiment: representation-quality supporting task)

Builds the ~10k-case induced subgraph and per-relation train/val/test triple
splits used by `link_prediction_rgcn.ipynb`, `link_prediction_graphsage.ipynb`,
and `link_prediction_transe.ipynb`. Evaluates three curated relations
(Vehicle/Account/Task -- see "Extract curated relations" below), not
general/automatic relation discovery.

**Run once per schema, in separate kernel sessions** -- set `SCHEMA_LABEL`,
run the notebook, restart the kernel, flip it, run again. The first run
samples ~10k shared case ids and saves them; the second run reuses that
exact list so both schemas share identical cases.

**Inputs required**: `{volume}base_vars.pt`/`sulo_vars.pt` and the raw
`.nt` instance file for the selected schema. Optionally
`{volume}graph_geometry/rq1a_structural_comparison.csv` for an informational
hop-distance sanity check.

In [14]:
%restart_python

In [15]:
# Mitigates CUDA memory fragmentation, and (more importantly here) clears
# any GPU memory a PREVIOUS kernel/process on this cluster left allocated --
# a real, reproduced OOM in link_prediction_rgcn.ipynb turned out to be
# exactly this: the GPU already had 15.45/15.56 GiB in use by a leftover
# process before this notebook's own first tensor.to(device) call even ran.
# Must run before any CUDA context is created, right after the restart above.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
import os
import sys

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import torch
import pandas as pd

from training_data import Ontology
from link_prediction.link_prediction_tools import (
    sample_case_ids,
    resolve_case_node_ids,
    build_induced_subgraph,
    resolve_relation_triples,
    get_relation_edge_type_ids,
    split_relation_triples,
    split_train_disjoint,
    mask_held_out_triples,
    mask_role_chain_nodes,
    build_candidate_pool,
    build_true_tails_by_head,
)

In [ ]:
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"

sulo_data_path = f"{volume}subgraph_balanced_sulo_populated_graph.nt"
baseline_data_path = f"{volume}subset_balanced_baseline_populated_graph.nt"

BASELINE_VARS = f"{volume}base_vars.pt"
SULO_VARS = f"{volume}sulo_vars.pt"

LP_DIR = f"{volume}link_prediction/"
os.makedirs(LP_DIR, exist_ok=True)

CASE_IDS_PATH = f"{LP_DIR}sampled_case_ids.pt"

N_CASES = 10000
NUM_HOPS = 2

# Run once per schema, in separate kernel sessions -- restart and flip
# SCHEMA_LABEL for the second run.
SCHEMA_LABEL = 'base'  # <<< CHANGE TO 'sulo' FOR THE SECOND RUN

# Caps TRAINING SUPERVISION triples per relation only (val/test are never
# capped, so reported metrics stay full-coverage) -- per-epoch cost scales
# directly with this count. Does NOT affect TransE, which trains on
# sub_graph_data's real topology directly, not train_sup_triples_by_relation.
# Changing this requires re-running this notebook for BOTH schemas and
# re-running every downstream LP notebook's dimension search, since it's
# baked into the saved {schema}_lp_data.pt file.
MAX_TRAIN_SUP_PER_RELATION = 8000

## Load `{schema}_vars.pt` + re-parse Ontology

`graph_data`/`vector_table` load straight from RGCN.ipynb/graph_sage.ipynb's
cached artifacts. The `Ontology` re-parse is only needed for
`enum_nodes`/`enum_relations` to resolve Case/relation names.

In [18]:
data_path = baseline_data_path if SCHEMA_LABEL == 'base' else sulo_data_path
vars_path = BASELINE_VARS if SCHEMA_LABEL == 'base' else SULO_VARS
label_predicate = "hasSurveyRating" if SCHEMA_LABEL == 'base' else "hasFeature"

variables = torch.load(vars_path, weights_only=False)
graph_data_full = variables[f'{SCHEMA_LABEL}_graph_data']
vector_table = variables[f'{SCHEMA_LABEL}_vector_table']

# Cache-first: only enum_relations is used from this Ontology parse (see the
# id_to_relation_uri build below), so the shared {schema}_ontology_enums.pt
# cache (written by graph_sage.ipynb/RGCN.ipynb/extract_graph_geometry.ipynb)
# covers it without touching the raw .nt file again.
_enum_cache_path = f"{volume}{SCHEMA_LABEL}_ontology_enums.pt"
if os.path.exists(_enum_cache_path):
    from types import SimpleNamespace
    _enum_cache = torch.load(_enum_cache_path, map_location='cpu', weights_only=False)
    ontology_obj = SimpleNamespace(
        enum_nodes=_enum_cache['enum_nodes'],
        enum_relations=_enum_cache['enum_relations'],
    )
    print(f"[{SCHEMA_LABEL}] Loaded cached enum_nodes/enum_relations from {_enum_cache_path} "
          f"-- no .nt file access needed.")
else:
    ontology_obj = Ontology(
        nt_file_path=data_path,
        label_predicate=label_predicate,
        label_object_pattern=r"SurveyRating/(\d+)",
    )

[base] Loaded cached enum_nodes/enum_relations from /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/base_ontology_enums.pt -- no .nt file access needed.

## Sample or reuse shared Case ids

Cases are sampled freely (satisfaction labels aren't used for link
prediction). Whichever schema runs first samples `N_CASES` ids and saves
them to `CASE_IDS_PATH`; the second run reuses that exact list so both
schemas' subgraphs are built around identical cases.

In [19]:
if os.path.exists(CASE_IDS_PATH):
    case_ids = torch.load(CASE_IDS_PATH, weights_only=False)
    print(f"Loaded {len(case_ids)} case ids sampled during the other schema's run "
          f"({CASE_IDS_PATH}).")
    case_ids, case_node_ids = resolve_case_node_ids(graph_data_full, ontology_obj, SCHEMA_LABEL, case_ids)
else:
    case_ids, case_node_ids = sample_case_ids(graph_data_full, ontology_obj, SCHEMA_LABEL, n=N_CASES, seed=42)
    torch.save(case_ids, CASE_IDS_PATH)
    print(f"Sampled {len(case_ids)} case ids, saved to {CASE_IDS_PATH} for the other "
          f"schema's run to reuse.")

Loaded 10000 case ids sampled during the other schema's run (/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/link_prediction/sampled_case_ids.pt).

## Build induced subgraph

In [20]:
sub_data, local_to_orig, orig_to_local = build_induced_subgraph(
    graph_data_full, case_node_ids, num_hops=NUM_HOPS,
)

Induced LP subgraph: 1,128,708 nodes, 6,299,367 edges (from 10,000 seed cases, 2-hop expansion).

## Extract curated relations (Vehicle / Account / Task only)

Three relations are evaluated as `(h, r, ?)` targets: Vehicle, Account,
Task -- real many-to-one entity relations. Text relations
(CaseDescription/TaskDescription/CaseSolutionFinalStatement) are excluded:
their candidate pools are near-1:1 with cases, a much less meaningful
ranking problem.

Each schema populates these differently, via its own `RELATION_SPECS`
entry: **Base** uses direct edges (`hasVehicle`/`hasAccount`/`hasTask`).
**SULO** reifies Vehicle/Account through its Role pattern
(`hasParticipant`->Role->`isFeatureOf`->target); Task uses a direct edge
(`hasPart`, shared with CaseModification, filtered by target concept).
`resolve_relation_triples` dispatches on each spec's `type`.

In [21]:
RELATION_SPECS = {
    'base': {
        'Vehicle': {'type': 'direct', 'predicate': 'hasVehicle', 'target_concept': 'Vehicle'},
        'Account': {'type': 'direct', 'predicate': 'hasAccount', 'target_concept': 'Account'},
        'Task':    {'type': 'direct', 'predicate': 'hasTask', 'target_concept': 'Task'},
    },
    'sulo': {
        'Vehicle': {'type': 'role_chain', 'to_role_predicate': 'hasParticipant', 'role_concept': 'VehicleRole',
                    'from_role_predicate': 'isFeatureOf', 'target_concept': 'Vehicle'},
        'Account': {'type': 'role_chain', 'to_role_predicate': 'hasParticipant', 'role_concept': 'AccountRole',
                    'from_role_predicate': 'isFeatureOf', 'target_concept': 'Account'},
        'Task':    {'type': 'direct', 'predicate': 'hasPart', 'target_concept': 'Task'},
    },
}

RELATION_NAMES = list(RELATION_SPECS[SCHEMA_LABEL].keys())
role_chain_relation_names = {name for name, spec in RELATION_SPECS[SCHEMA_LABEL].items()
                              if spec['type'] == 'role_chain'}

raw_triples_by_relation = {}
raw_aux_by_relation = {}
for relation_name, spec in RELATION_SPECS[SCHEMA_LABEL].items():
    triples, aux = resolve_relation_triples(
        sub_data, ontology_obj, SCHEMA_LABEL, spec, graph_data_full, local_to_orig,
    )
    raw_triples_by_relation[relation_name] = triples
    raw_aux_by_relation[relation_name] = aux
    print(f"[{SCHEMA_LABEL}] {relation_name} ({spec['type']}): {triples.size(0)} triples in induced subgraph")

[base] Vehicle (direct): 8164 triples in induced subgraph
[base] Account (direct): 10981 triples in induced subgraph
[base] Task (direct): 118658 triples in induced subgraph

## Extract & split relations

For each curated relation: build the candidate pool + true-tail lookup
from the full triple set, then 80/10/10-split. Role-chain relations carry
their Role node id as `aux` through the same split, for
`mask_role_chain_nodes` later.

In [22]:
all_triples_by_relation = {}
split_by_relation = {}
candidate_pools = {}
true_tails_by_relation = {}

for relation_name in RELATION_NAMES:
    triples = raw_triples_by_relation[relation_name]
    aux = raw_aux_by_relation[relation_name]

    all_triples_by_relation[relation_name] = triples
    candidate_pools[relation_name] = build_candidate_pool(triples)
    true_tails_by_relation[relation_name] = build_true_tails_by_head(triples)

    split = split_relation_triples(triples, aux=aux, val_frac=0.1, test_frac=0.1, seed=42)
    split_by_relation[relation_name] = split

    n = triples.size(0)
    pool_size = candidate_pools[relation_name].size(0)
    status = "split ok" if split is not None else "SKIPPED (too few triples)"
    print(f"[{SCHEMA_LABEL}] {relation_name}: {n} triples, {pool_size} candidate tails -- {status}")

[base] Vehicle: 8164 triples, 107 candidate tails -- split ok
[base] Account: 10981 triples, 10290 candidate tails -- split ok
[base] Task: 118658 triples, 118658 candidate tails -- split ok

## Cross-check hop-distance against Table 1 (informational)

Sanity check against `extract_graph_geometry.ipynb`'s
`rq1a_structural_comparison.csv` -- a large Ad-hoc/SULO gap in
mean-hops-to-Case flags a relation that isn't a like-for-like comparison.
Optional.

In [23]:
rq1a_path = f"{volume}graph_geometry/rq1a_structural_comparison.csv"
if os.path.exists(rq1a_path):
    rq1a_df = pd.read_csv(rq1a_path, index_col='metric')
    concept_names = ['CaseDescription', 'TaskDescription', 'CaseSolutionFinalStatement',
                      'Vehicle', 'Account', 'Task']
    print("Hop-distance sanity check -- large Ad-hoc/SULO gaps flag a relation that")
    print("may not be a like-for-like 1-hop comparison between the two KGs:")
    for concept in concept_names:
        row_key = f'{concept}_mean_hops_to_case'
        if row_key in rq1a_df.index:
            base_hops = float(rq1a_df.loc[row_key, 'Ad-hoc (Base)'])
            sulo_hops = float(rq1a_df.loc[row_key, 'SULO'])
            flag = "  <-- DIFFERS" if abs(base_hops - sulo_hops) > 0.5 else ""
            print(f"  {concept}: Ad-hoc={base_hops:.2f}, SULO={sulo_hops:.2f}{flag}")
        else:
            print(f"  {concept}: not found in {rq1a_path}")
else:
    print(f"[missing] {rq1a_path} -- run extract_graph_geometry.ipynb first to enable "
          f"this sanity check (optional, not required to proceed).")

Hop-distance sanity check -- large Ad-hoc/SULO gaps flag a relation that
may not be a like-for-like 1-hop comparison between the two KGs:
  CaseDescription: Ad-hoc=1.00, SULO=1.00
  TaskDescription: Ad-hoc=1.97, SULO=1.99
  CaseSolutionFinalStatement: Ad-hoc=1.00, SULO=1.00
  Vehicle: Ad-hoc=1.70, SULO=2.60  <-- DIFFERS
  Account: Ad-hoc=1.00, SULO=2.00  <-- DIFFERS
  Task: Ad-hoc=1.02, SULO=1.04

## Mask held-out triples + resolve relation forward-edge ids

val+test triples are removed, producing `masked_sub_data` (saved as
`sub_graph_data`, what `link_prediction_transe.ipynb` uses directly).

A `DISJOINT_TRAIN_FRAC` (0.3) slice of TRAIN triples is also carved out
into a further-masked copy, `mp_sub_graph_data` -- the message-passing
graph RGCN/GraphSAGE train against, so a train triple is never
simultaneously a message-passing edge and a supervision target (see
`link_prediction_tools.split_train_disjoint`). That slice is capped at
`MAX_TRAIN_SUP_PER_RELATION`; val/test are never capped.

Masking differs by relation type: **DIRECT** relations go through
`mask_held_out_triples` (keyed by predicate name); **ROLE_CHAIN** relations
go through `mask_role_chain_nodes` (severing the intermediate Role node,
since there's no single edge to remove for a two-hop reified relation).

`relation_fwd_ids` maps each relation to the DistMult decoder's embedding
row: DIRECT relations reuse their real edge_type id, ROLE_CHAIN relations
get a synthetic id appended after the real relation id space (same scheme
`link_prediction_transe.ipynb` uses for its pykeen factory).

In [24]:
held_out_by_predicate = {}          # DIRECT: predicate name -> (h, t) pairs to remove (val+test)
held_out_role_node_ids_list = []    # ROLE_CHAIN: intermediate Role node ids to sever (val+test)

train_sup_by_relation = {}
train_sup_by_predicate = {}         # DIRECT: predicate name -> (h, t) pairs to remove (capped train_sup)
train_sup_role_node_ids_list = []   # ROLE_CHAIN: intermediate Role node ids to sever (capped train_sup)
train_sup_capped_from = {}          # relation_name -> pre-cap size, only for relations actually capped

_empty_pairs = torch.empty((0, 2), dtype=torch.long)
_empty_aux = torch.empty(0, dtype=torch.long)

# 0.3 matches PyG RandomLinkSplit's disjoint_train_ratio convention.
DISJOINT_TRAIN_FRAC = 0.3

_cap_generator = torch.Generator().manual_seed(42)

for relation_name in RELATION_NAMES:
    split = split_by_relation[relation_name]
    if split is None:
        continue
    spec = RELATION_SPECS[SCHEMA_LABEL][relation_name]
    is_role_chain = spec['type'] == 'role_chain'

    val_test_triples = torch.cat([split['val'], split['test']], dim=0)
    if is_role_chain:
        val_test_aux = torch.cat([split['val_aux'], split['test_aux']], dim=0)
        held_out_role_node_ids_list.append(val_test_aux)
    else:
        held_out_by_predicate[spec['predicate']] = torch.cat(
            [held_out_by_predicate.get(spec['predicate'], _empty_pairs), val_test_triples], dim=0,
        )

    train_aux = split.get('train_aux') if is_role_chain else None
    train_split = split_train_disjoint(split['train'], aux=train_aux, frac=DISJOINT_TRAIN_FRAC)
    if train_split is not None:
        train_sup = train_split['sup']
        train_sup_aux = train_split.get('sup_aux') if is_role_chain else None
    else:
        train_sup = torch.empty((0, 2), dtype=torch.long)
        train_sup_aux = _empty_aux if is_role_chain else None

    # Cap training SUPERVISION only (val/test untouched). Random seeded
    # subsample, not "first N", to avoid order bias.
    if train_sup.size(0) > MAX_TRAIN_SUP_PER_RELATION:
        train_sup_capped_from[relation_name] = train_sup.size(0)
        keep_idx = torch.randperm(train_sup.size(0), generator=_cap_generator)[:MAX_TRAIN_SUP_PER_RELATION]
        train_sup = train_sup[keep_idx]
        if is_role_chain:
            train_sup_aux = train_sup_aux[keep_idx]

    train_sup_by_relation[relation_name] = train_sup
    if train_sup.size(0) > 0:
        if is_role_chain:
            train_sup_role_node_ids_list.append(train_sup_aux)
        else:
            train_sup_by_predicate[spec['predicate']] = torch.cat(
                [train_sup_by_predicate.get(spec['predicate'], _empty_pairs), train_sup], dim=0,
            )

if train_sup_capped_from:
    print(f"[{SCHEMA_LABEL}] Capped {len(train_sup_capped_from)} relation(s) to "
          f"{MAX_TRAIN_SUP_PER_RELATION:,} training-supervision triples each "
          f"(val/test untouched):")
    for relation_name, original_n in sorted(train_sup_capped_from.items(), key=lambda kv: -kv[1]):
        print(f"  {relation_name}: {original_n:,} -> {MAX_TRAIN_SUP_PER_RELATION:,}")

held_out_role_node_ids = (torch.cat(held_out_role_node_ids_list) if held_out_role_node_ids_list
                           else _empty_aux)
train_sup_role_node_ids = (torch.cat(train_sup_role_node_ids_list) if train_sup_role_node_ids_list
                            else _empty_aux)

# masked_sub_data: val+test removed only (saved below as sub_graph_data,
# what link_prediction_transe.ipynb needs untouched).
masked_sub_data = mask_held_out_triples(sub_data, ontology_obj, held_out_by_predicate)
masked_sub_data = mask_role_chain_nodes(masked_sub_data, held_out_role_node_ids)

# mp_sub_graph_data: masked_sub_data with the capped train-supervision
# slice also removed, as a separate pass so the shared sub_graph_data/
# train_triples_by_relation keys TransE depends on stay untouched.
mp_sub_graph_data = (mask_held_out_triples(masked_sub_data, ontology_obj, train_sup_by_predicate)
                      if train_sup_by_predicate else masked_sub_data.clone())
mp_sub_graph_data = mask_role_chain_nodes(mp_sub_graph_data, train_sup_role_node_ids)

# relation_fwd_ids: DistMult decoder embedding-row index per relation.
# DIRECT relations reuse their real (forward/even) edge_type id. ROLE_CHAIN
# relations have no real edge_type (a two-hop chain of two predicates), so
# they get a synthetic id appended after the real relation id space -- same
# scheme link_prediction_transe.ipynb uses for its pykeen factory.
_num_real_relations = (int(masked_sub_data.edge_type.max().item()) + 1
                        if masked_sub_data.edge_index.size(1) > 0 else 0)
relation_fwd_ids = {}
_next_synthetic_id = _num_real_relations
for relation_name in RELATION_NAMES:
    if split_by_relation[relation_name] is None:
        continue
    spec = RELATION_SPECS[SCHEMA_LABEL][relation_name]
    if spec['type'] == 'role_chain':
        relation_fwd_ids[relation_name] = _next_synthetic_id
        _next_synthetic_id += 1
    else:
        ids = get_relation_edge_type_ids(ontology_obj, spec['predicate'])
        relation_fwd_ids[relation_name] = ids[0][0]  # forward id -- ids[0] is the fwd_ids list

decoder_num_relations = _next_synthetic_id
print(f"[{SCHEMA_LABEL}] decoder_num_relations={decoder_num_relations}")

[base] Capped 1 relation(s) to 8,000 training-supervision triples each (val/test untouched):
  Task: 28,478 -> 8,000
Masked 52,318 held-out direct-relation edges (of 6,299,367).
Masked 24,136 held-out direct-relation edges (of 6,247,049).
[base] decoder_num_relations=76

## Save LP data bundle

`vector_table` is sliced to only the rows this subgraph references before
saving -- keeping the full table GPU-resident caused a real, reproduced
OOM in `link_prediction_rgcn.ipynb` (identical failure at every
hidden_dim, pointing at the per-epoch full-table gather, not model size).

In [25]:
used_rows = torch.unique(masked_sub_data.n_id_to_vec_row)
row_remap = torch.full((vector_table.shape[0],), -1, dtype=torch.long)
row_remap[used_rows] = torch.arange(used_rows.size(0))

sliced_vector_table = vector_table[used_rows].clone()
masked_sub_data.n_id_to_vec_row = row_remap[masked_sub_data.n_id_to_vec_row]
# mp_sub_graph_data has the identical node set/id space as masked_sub_data
# (masking only ever removes edges, never nodes) -- same row_remap applies.
mp_sub_graph_data.n_id_to_vec_row = row_remap[mp_sub_graph_data.n_id_to_vec_row]

print(f"Sliced vector_table: {vector_table.shape[0]:,} -> {sliced_vector_table.shape[0]:,} rows "
      f"({sliced_vector_table.shape[0] / vector_table.shape[0] * 100:.1f}% of the full table).")

empty = torch.empty((0, 2), dtype=torch.long)
save_path = f"{LP_DIR}{SCHEMA_LABEL}_lp_data.pt"
torch.save({
    # UNCHANGED keys -- link_prediction_transe.ipynb builds its
    # TriplesFactory directly from these (see the "Mask held-out triples"
    # section's markdown for why it must keep seeing every train triple).
    'sub_graph_data': masked_sub_data,
    'train_triples_by_relation': {r: (s['train'] if s is not None else empty)
                                   for r, s in split_by_relation.items()},
    # NEW keys -- link_prediction_rgcn.ipynb/link_prediction_graphsage.ipynb
    # use these instead: mp_sub_graph_data has the disjoint train-
    # supervision slice removed from message passing, and
    # train_sup_triples_by_relation is that removed slice (the ONLY train
    # triples supervised, so training never sees the exact edge it's being
    # asked to predict).
    'mp_sub_graph_data': mp_sub_graph_data,
    'train_sup_triples_by_relation': {r: train_sup_by_relation.get(r, empty) for r in split_by_relation},
    'vector_table': sliced_vector_table,
    'val_triples_by_relation': {r: (s['val'] if s is not None else empty)
                                 for r, s in split_by_relation.items()},
    'test_triples_by_relation': {r: (s['test'] if s is not None else empty)
                                  for r, s in split_by_relation.items()},
    'candidate_pools': candidate_pools,
    'true_tails_by_relation': true_tails_by_relation,
    'relation_fwd_ids': relation_fwd_ids,
    'role_chain_relation_names': role_chain_relation_names,
    'decoder_num_relations': decoder_num_relations,
}, save_path)
print(f"Saved LP data bundle: {save_path}")

Sliced vector_table: 2,281,381 -> 490,977 rows (21.5% of the full table).
Saved LP data bundle: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/link_prediction/base_lp_data.pt

## Final sanity summary

In [26]:
print(f"\n=== {SCHEMA_LABEL.upper()} LP data summary ===")
print(f"sub_graph_data (val/test masked, for TransE): {masked_sub_data.num_nodes:,} nodes, "
      f"{masked_sub_data.edge_index.size(1):,} edges")
print(f"mp_sub_graph_data (val/test/train_sup masked, for RGCN/GraphSAGE): "
      f"{mp_sub_graph_data.num_nodes:,} nodes, {mp_sub_graph_data.edge_index.size(1):,} edges")
for relation_name in RELATION_NAMES:
    split = split_by_relation[relation_name]
    pool_size = candidate_pools[relation_name].size(0)
    if split is None:
        print(f"  {relation_name}: SKIPPED (insufficient triples)")
    else:
        n_train = split['train'].size(0)
        n_sup = train_sup_by_relation[relation_name].size(0)
        print(f"  {relation_name}: train_mp={n_train - n_sup}, train_sup={n_sup}, "
              f"val={split['val'].size(0)}, test={split['test'].size(0)}, "
              f"candidate_pool={pool_size}")

_other_schema = 'base' if SCHEMA_LABEL == 'sulo' else 'sulo'
_other_path = f"{LP_DIR}{_other_schema}_lp_data.pt"
if os.path.exists(_other_path):
    print(f"\nBoth schemas' LP data bundles are ready ({LP_DIR}base_lp_data.pt, "
          f"{LP_DIR}sulo_lp_data.pt) -- proceed to the model notebooks.")
else:
    print(f"\nRestart the kernel, set SCHEMA_LABEL = '{_other_schema}' above, and re-run "
          f"this notebook to build the other schema's LP data.")

=== BASE LP data summary ===
sub_graph_data (val/test masked, for TransE): 1,128,708 nodes, 6,247,049 edges
mp_sub_graph_data (val/test/train_sup masked, for RGCN/GraphSAGE): 1,128,708 nodes, 6,222,913 edges
  Vehicle: train_mp=4573, train_sup=1959, val=816, test=816, candidate_pool=107
  Account: train_mp=6150, train_sup=2635, val=1098, test=1098, candidate_pool=10290
  Task: train_mp=86928, train_sup=8000, val=11865, test=11865, candidate_pool=118658

Both schemas' LP data bundles are ready (/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/link_prediction/base_lp_data.pt, /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/link_prediction/sulo_lp_data.pt) -- proceed to the model notebooks.